# Open in Colab — 3D Craniofacial Pipeline

Original experiment design, scaled to many CT/CBCT files.

For every scan: Run 1 = segment → project → record original metrics. Run 2 repeats the same pipeline independently.

Then:
1. Same person: Run 1 vs Run 2 for repeatability.
2. Different people: unique unordered pairs only, following the original between-case comparison logic. For 50 distinct people this is 1,225 pairs.


## 1. Runtime check

Select **Runtime → Change runtime type → GPU** first. The assigned GPU is recorded.

In [ ]:
import os, sys, platform, subprocess
print("Python:", sys.version.split()[0])
print("Platform:", platform.platform())
try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
        print("VRAM GiB:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 2))
except Exception as e:
    print("PyTorch not ready yet:", e)
subprocess.run(["nvidia-smi"], check=False)

## 2. Clone and install

The runner uses the repository's pinned TotalSegmentator version and the existing `fast=False` settings.

In [ ]:
!rm -rf /content/3D-Craniofacial-Pipeline
!git clone -q -b feat/colab-runner https://github.com/trungnb/3D-Craniofacial-Pipeline.git /content/3D-Craniofacial-Pipeline
!python -m pip install -q --upgrade pip
!python -m pip install -q -r /content/3D-Craniofacial-Pipeline/requirements.txt
import totalsegmentator, torch
print("TotalSegmentator:", getattr(totalsegmentator, "__version__", "installed"))
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available(): print("GPU:", torch.cuda.get_device_name(0))

## 3. Google Drive + input data

Put the CT/CBCT files in `My Drive/new_data/`.

If multiple scans are from the **same person**, put them in one subfolder, e.g. `new_data/person01/scan1.nii.gz` and `new_data/person01/scan2.nii.gz`. Same-person pairs are skipped.

If all files are directly inside `new_data/`, each file is treated as a different person.

In [ ]:
from google.colab import drive
from pathlib import Path
import shutil, time, json, itertools
from datetime import datetime, timezone
from collections import Counter

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive")
DATA_DIR = DRIVE_ROOT / "new_data"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
RUN_ROOT = DRIVE_ROOT / "new_data_runs" / RUN_ID
LOCAL_ROOT = Path("/content/cbct_pipeline")
LOCAL_INPUT = LOCAL_ROOT / "input"
LOCAL_OUTPUT = LOCAL_ROOT / "output"

if not DATA_DIR.exists():
    raise FileNotFoundError(f"Create {DATA_DIR} and add .nii/.nii.gz files first.")

nii_files = sorted(p for p in DATA_DIR.rglob("*") if p.is_file() and (p.name.endswith(".nii") or p.name.endswith(".nii.gz")))
if not nii_files:
    raise FileNotFoundError(f"No NIfTI files found under {DATA_DIR}.")

cases = []
for src in nii_files:
    rel = src.relative_to(DATA_DIR)
    person_id = src.stem.replace(".nii", "") if len(rel.parts) == 1 else rel.parts[0]
    scan_id = src.name.replace(".nii.gz", "").replace(".nii", "")
    cases.append({"source": src, "person_id": person_id, "scan_id": scan_id})

print("Found", len(cases), "scan(s)")
print("Person groups:")
for person, n in Counter(c["person_id"] for c in cases).items():
    print(f"  {person}: {n} scan(s)")

## 4. Run the original pipeline twice

Each pass processes one file at a time: TotalSegmentator → projections → timing/storage metrics → clear local scratch. Run 2 is a second full pass over the same inputs.


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from totalsegmentator.python_api import totalsegmentator

def format_size(bytes_size):
    for unit in ['B','KB','MB','GB','TB']:
        if bytes_size < 1024:
            return f'{bytes_size:.2f} {unit}'
        bytes_size /= 1024
    return f'{bytes_size:.2f} PB'

def folder_size(path):
    return sum(p.stat().st_size for p in Path(path).rglob('*') if p.is_file())

def segmentation_function(input_path, output_root):
    basename = Path(input_path).name.replace('.nii.gz','').replace('.nii','')
    output_dir = Path(output_root) / basename
    output_dir.mkdir(parents=True, exist_ok=True)
    log_file = output_dir / 'segmentation_log.txt'
    start_str, start = time.ctime(), time.time()
    print(f'\nStarting TotalSegmentator for {basename}')
    totalsegmentator(input=str(input_path), output=str(output_dir), task='craniofacial_structures', output_type='niftis', fast=False)
    totalsegmentator(input=str(input_path), output=str(output_dir), task='teeth', output_type='niftis', fast=False)
    elapsed = time.time() - start
    with open(log_file, 'w') as f:
        f.write(f'Segmentation for: {basename}\nStart Time: {start_str}\nEnd Time: {time.ctime()}\nElapsed Time: {elapsed:.2f} seconds\n')
    return elapsed

def base_projection_function(input_path, output_path):
    img = nib.load(str(input_path))
    mask = (img.get_fdata() > 0).astype(np.uint8)
    out = Path(output_path)
    out.mkdir(parents=True, exist_ok=True)
    plt.imsave(out/'axial_projection.png', np.max(mask, axis=2), cmap='gray')
    plt.imsave(out/'coronal_projection.png', np.max(mask, axis=1), cmap='gray')
    plt.imsave(out/'sagital_projection.png', np.max(mask, axis=0), cmap='gray')

def complete_projection_function(person, volumes_root, projections_root):
    person_vol = Path(volumes_root) / person
    person_pro = Path(projections_root) / person
    person_pro.mkdir(parents=True, exist_ok=True)
    start_str, start = time.ctime(), time.time()
    for volume in sorted(person_vol.iterdir()):
        if volume.name.endswith('.nii.gz') or volume.name.endswith('.nii'):
            name = volume.name.replace('.nii.gz','').replace('.nii','')
            base_projection_function(volume, person_pro/name)
    elapsed = time.time() - start
    with open(person_pro/'projection_log.txt', 'w') as f:
        f.write(f'Projection for: {person}\nStart Time: {start_str}\nEnd Time: {time.ctime()}\nElapsed Time: {elapsed:.2f} seconds\n')
    return elapsed

def run_pass(pass_name):
    run_root = EXPERIMENT_ROOT / pass_name
    volumes_root, projections_root = run_root/'Volumes', run_root/'Projections'
    run_root.mkdir(parents=True, exist_ok=False)
    rows = []
    for case in cases:
        src, person, scan = case['source'], case['person_id'], case['scan_id']
        print(f'\n===== {pass_name}: {scan} | person={person} =====')
        shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
        local_input = LOCAL_ROOT/'input'
        local_input.mkdir(parents=True, exist_ok=True)
        local_src = local_input/src.name
        shutil.copy2(src, local_src)
        seg_time = segmentation_function(local_src, volumes_root)
        proj_time = complete_projection_function(scan, volumes_root, projections_root)
        volume_bytes = folder_size(volumes_root/scan)
        projection_bytes = folder_size(projections_root/scan)
        rows.append({
            'person':person, 'scan':scan,
            'ct_size_bytes':src.stat().st_size, 'ct_size':format_size(src.stat().st_size),
            'segmentation_time (s)':seg_time,
            'total_volume_size_bytes':volume_bytes, 'total_volume_size':format_size(volume_bytes),
            'projection_time (s)':proj_time,
            'total_projection_size_bytes':projection_bytes, 'total_projection_size':format_size(projection_bytes)
        })
        shutil.rmtree(LOCAL_ROOT, ignore_errors=True)
    df = pd.DataFrame(rows)
    df.to_csv(run_root/'summary_report.csv', index=False)
    return df

RUN1_DF = run_pass('first_run')


## 5. Run 2

Run the same complete pipeline again on the same input files. No comparison is substituted for this pass.


In [ ]:
RUN2_DF = run_pass('second_run')
print('Run 1:', len(RUN1_DF), 'scan(s)')
print('Run 2:', len(RUN2_DF), 'scan(s)')


## 6. Same-person repeatability — Run 1 vs Run 2

This preserves the original consistency test: corresponding outputs from the two runs are compared with Dice/IoU.


In [ ]:
import SimpleITK as sitk
import cv2

def score_metrics(mask1, mask2):
    mask1, mask2 = mask1.astype(bool), mask2.astype(bool)
    inter = np.sum(mask1 & mask2)
    a, b = np.sum(mask1), np.sum(mask2)
    dice = 1.0 if a == 0 and b == 0 else (0.0 if a == 0 or b == 0 else 2*inter/(a+b))
    union = np.sum(mask1 | mask2)
    iou = 1.0 if union == 0 else inter/union
    return float(dice), float(iou)

def compare_nii(path1, path2):
    img1, img2 = sitk.ReadImage(str(path1)), sitk.ReadImage(str(path2))
    img2 = sitk.Resample(img2, img1, sitk.Transform(), sitk.sitkNearestNeighbor, 0, img2.GetPixelID())
    return score_metrics(sitk.GetArrayFromImage(img1)>0, sitk.GetArrayFromImage(img2)>0)

def compare_png(path1, path2):
    img1, img2 = cv2.imread(str(path1), cv2.IMREAD_GRAYSCALE), cv2.imread(str(path2), cv2.IMREAD_GRAYSCALE)
    if img1 is None or img2 is None: raise ValueError('Cannot read projection PNG')
    mask1, mask2 = (img1>0).astype(np.uint8), (img2>0).astype(np.uint8)
    if mask1.shape != mask2.shape:
        mask2 = cv2.resize(mask2, (mask1.shape[1], mask1.shape[0]), interpolation=cv2.INTER_NEAREST)
    return score_metrics(mask1,mask2)

same_vol_rows, same_pro_rows = [], []
for case in cases:
    person, scan = case['person_id'], case['scan_id']
    v1 = EXPERIMENT_ROOT/'first_run'/'Volumes'/scan
    v2 = EXPERIMENT_ROOT/'second_run'/'Volumes'/scan
    for fname in sorted(set(p.name for p in v1.glob('*.nii.gz')) & set(p.name for p in v2.glob('*.nii.gz'))):
        t=time.time()
        dice,iou=compare_nii(v1/fname,v2/fname)
        same_vol_rows.append({'person':person,'scan':scan,'volume':fname,'dice_score':dice,'iou_score':iou,'time':time.time()-t})
    p1 = EXPERIMENT_ROOT/'first_run'/'Projections'/scan
    p2 = EXPERIMENT_ROOT/'second_run'/'Projections'/scan
    for volume_name in sorted(set(p.name for p in p1.iterdir() if p.is_dir()) & set(p.name for p in p2.iterdir() if p.is_dir())):
        for fname in sorted(set(p.name for p in (p1/volume_name).glob('*.png')) & set(p.name for p in (p2/volume_name).glob('*.png'))):
            t=time.time()
            dice,iou=compare_png(p1/volume_name/fname,p2/volume_name/fname)
            same_pro_rows.append({'person':person,'scan':scan,'volume':volume_name,'projection':fname,'dice_score':dice,'iou_score':iou,'time':time.time()-t})

same_vol = pd.DataFrame(same_vol_rows)
same_pro = pd.DataFrame(same_pro_rows)
same_vol.to_csv(EXPERIMENT_ROOT/'same_person_volume_consistency.csv',index=False)
same_pro.to_csv(EXPERIMENT_ROOT/'same_person_projection_consistency.csv',index=False)
display(same_vol.head())
display(same_pro.head())


## 7. Different-person comparison — unique pairs only

This preserves the original between-case experiment by using Run 1 only. itertools.combinations evaluates each unordered pair once, so A vs B is not repeated later as B vs A.

With 50 distinct people: 50 × 49 / 2 = 1,225 unique pairs.


In [ ]:
scan_ids = sorted(c['scan_id'] for c in cases)
people = {c['scan_id']:c['person_id'] for c in cases}
different_vol_rows, different_pro_rows = [], []

for scan1, scan2 in itertools.combinations(scan_ids, 2):
    if people[scan1] == people[scan2]:
        continue
    v1 = EXPERIMENT_ROOT/'first_run'/'Volumes'/scan1
    v2 = EXPERIMENT_ROOT/'first_run'/'Volumes'/scan2
    for fname in sorted(set(p.name for p in v1.glob('*.nii.gz')) & set(p.name for p in v2.glob('*.nii.gz'))):
        t=time.time()
        dice,iou=compare_nii(v1/fname,v2/fname)
        different_vol_rows.append({'person1':people[scan1],'person2':people[scan2],'scan1':scan1,'scan2':scan2,'volume':fname,'dice_score':dice,'iou_score':iou,'time':time.time()-t})
    p1 = EXPERIMENT_ROOT/'first_run'/'Projections'/scan1
    p2 = EXPERIMENT_ROOT/'first_run'/'Projections'/scan2
    common_volumes = sorted(set(p.name for p in p1.iterdir() if p.is_dir()) & set(p.name for p in p2.iterdir() if p.is_dir()))
    for volume_name in common_volumes:
        for fname in sorted(set(p.name for p in (p1/volume_name).glob('*.png')) & set(p.name for p in (p2/volume_name).glob('*.png'))):
            t=time.time()
            dice,iou=compare_png(p1/volume_name/fname,p2/volume_name/fname)
            different_pro_rows.append({'person1':people[scan1],'person2':people[scan2],'scan1':scan1,'scan2':scan2,'volume':volume_name,'projection':fname,'dice_score':dice,'iou_score':iou,'time':time.time()-t})

different_vol = pd.DataFrame(different_vol_rows)
different_pro = pd.DataFrame(different_pro_rows)
different_vol.to_csv(EXPERIMENT_ROOT/'different_person_volume_results.csv',index=False)
different_pro.to_csv(EXPERIMENT_ROOT/'different_person_projection_results.csv',index=False)

n_people = len(set(people.values()))
n_unique_pairs = sum(1 for a,b in itertools.combinations(sorted(set(people.values())),2))
print('People:', n_people)
print('Unique different-person pairs:', n_unique_pairs)
display(different_vol.head())
display(different_pro.head())


## 8. Original-style summaries


In [ ]:
same_summary = pd.DataFrame([
    {'comparison_type':'Volumes (same person Run1 vs Run2)','avg_dice_score':same_vol.dice_score.mean(),'avg_iou_score':same_vol.iou_score.mean(),'total_comparison_time_seconds':same_vol.time.sum()},
    {'comparison_type':'Projections (same person Run1 vs Run2)','avg_dice_score':same_pro.dice_score.mean(),'avg_iou_score':same_pro.iou_score.mean(),'total_comparison_time_seconds':same_pro.time.sum()}
])
different_summary = pd.DataFrame([
    {'comparison_type':'Volumes (different-person pairs, Run 1)','avg_dice_score':different_vol.dice_score.mean(),'avg_iou_score':different_vol.iou_score.mean(),'total_comparison_time_seconds':different_vol.time.sum()},
    {'comparison_type':'Projections (different-person pairs, Run 1)','avg_dice_score':different_pro.dice_score.mean(),'avg_iou_score':different_pro.iou_score.mean(),'total_comparison_time_seconds':different_pro.time.sum()}
])

same_summary.to_csv(EXPERIMENT_ROOT/'same_person_summary.csv',index=False)
different_summary.to_csv(EXPERIMENT_ROOT/'different_person_summary.csv',index=False)
run_summary = RUN1_DF.merge(RUN2_DF,on=['person','scan'],suffixes=('_run1','_run2'))
run_summary.to_csv(EXPERIMENT_ROOT/'run_summary.csv',index=False)

manifest = {
    'run_id':RUN_ID,
    'n_scans':len(cases),
    'n_persons':n_people,
    'n_unique_different_person_pairs':n_unique_pairs,
    'passes':['first_run','second_run'],
    'tasks':['craniofacial_structures','teeth'],
    'fast':False,
    'same_person_comparison':'Run 1 vs Run 2',
    'different_person_comparison':'Run 1 only; unique unordered pairs',
    '3d_comparison':'resample-to-reference + binary Dice/IoU',
    '2d_comparison':'nearest-neighbor resize when dimensions differ + binary Dice/IoU',
    'timestamp_utc':datetime.now(timezone.utc).isoformat()
}
(EXPERIMENT_ROOT/'run_manifest.json').write_text(json.dumps(manifest,indent=2))
display(run_summary.head())
display(same_summary)
display(different_summary)


## 9. Final design check

For 50 distinct people:
- Run 1: 50 complete segment → project passes.
- Run 2: the same 50 complete segment → project passes again.
- Same-person check: each person's Run 1 ↔ Run 2.
- Different-person check: unique pairs only, 1,225 pairs total.
- No reverse duplicate: A ↔ B is evaluated once.
- Different-person comparison uses Run 1 only, matching the original two-person between-case experiment.
- Metrics remain the original timing, storage, Dice and IoU measurements.
- No new registration, normalization, classifier, ROC/AUC, or other scientific layer is introduced.
